### 1. Check of the first results and first comparisons

It compares full model, remove of one attribute and activity zero on the four event logs. Using only the CSV of the sigle run: accuracy for categorical attributes and total MAE (`MAE_<attribute>`) for numerical ones.

This analisys checks the data, summarizes the 10 runs and compares each variant with the full model on the same targets. The tables are going to be located in `analysis/tables_activity_zero/`.

In [2]:
import re
import os 
import json
import numpy as np 
import pandas as pd

In [3]:
root_path = os.path.dirname(os.getcwd())
print(root_path)

/home/danbi/Projects/SANAGRAPH


In [35]:
results_dir = root_path + "/results_CAISE/"
tables_dir = root_path + "/analysis/tables/"

In [5]:
logs = [
    "bpi_2012_CZ",
    "bpi_2013_CZ",
    "sp2020_CZ",
    "BPI20_RequestForPayment_CZ"
]

In [6]:
patterns = ["odd", "even", "random", "window", "attr_level"]

In [7]:
num_runs = 10

with open(root_path + "/data/dataset_features.json") as f:
    dataset_features = json.load(f)
pd.set_option("display.max_columns", None)

In [8]:
dataset_features

{'BPI_Challenge_2013_open_problems': {'categorical': ['org:group',
   'resource country',
   'org:resource',
   'oranization country',
   'org:role',
   'Activity',
   'impact',
   'product',
   'org:resource:role'],
  'numerical': ['time:timestamp']},
 'sp2020': {'categorical': ['Activity',
   'REPAIR_IN_TIME_5D',
   'DEVICETYPE',
   'org:resource'],
  'numerical': ['time:timestamp']},
 'Helpdesk': {'categorical': ['Activity'], 'numerical': ['time:timestamp']},
 'BPI20_RequestForPayment': {'categorical': ['org:resource',
   'Activity',
   'org:role',
   'case:Project',
   'case:Task',
   'case:OrganizationalEntity',
   'case:Cost Type',
   'case:Activity',
   'case:RfpNumber',
   'org:resource:role'],
  'numerical': ['time:timestamp', 'case:RequestedAmount']},
 'BPI Challenge 2017 - Offer log': {'categorical': ['Action',
   'org:resource',
   'Activity',
   'EventOrigin',
   'lifecycle:transition',
   'case:Selected',
   'case:Accepted',
   'org:resource:role'],
  'numerical': ['time:

### 2. Check the results

For each configuration and pattern, check if the file exists.

List all the model and relative folder. For every dataset in log, it detects the categorical and numerical attributes from the `dataset_feature.json`. Then it prepares a row for every configuration:
- `full`: full model.
- `ablate::<feature>`: model trained without that feature.
- `Activity_zero`: full model but with Activity zeroed out.

The result is the configuration table.

In [ ]:
configurations = []

for log in logs:
    info = dataset_features[log]
    features = info["categorical"] + info["numerical"]

    # full model
    configurations.append({
        "log": log,
        "model": "full",
        "excluded_feature": "",
        "directory": results_dir + f"{log}/",
    })

    # ablation models
    for feature in features:
        safe_feature = re.sub(r"[^A-Za-z0-9_.-]+", "_", feature)

        configurations.append({
            "log": log,
            "model": f"ablate::{feature}",
            "excluded_feature": feature,
            "directory": (
                results_dir
                + f"ablation/{log}/ablation_{safe_feature}/"
            ),
        })

    # Activity-zero
    configurations.append({
        "log": log,
        "model": "Activity_zero",
        "excluded_feature": "Activity",
        "directory": (
            results_dir
            + f"activity-zero/{log}/ablation_Activity_zero/"
        ),
    })

configurations = pd.DataFrame(configurations)
configurations

,log,model,excluded_feature,directory
0,bpi_2012_CZ,full,,/home/danbi/Projects/SANAGRAPH/results_CAISE/b...
1,bpi_2012_CZ,ablate::Activity,Activity,/home/danbi/Projects/SANAGRAPH/results_CAISE/a...
2,bpi_2012_CZ,ablate::org:resource,org:resource,/home/danbi/Projects/SANAGRAPH/results_CAISE/a...
3,bpi_2012_CZ,ablate::concept:name,concept:name,/home/danbi/Projects/SANAGRAPH/results_CAISE/a...
4,bpi_2012_CZ,ablate::lifecycle:transition,lifecycle:transition,/home/danbi/Projects/SANAGRAPH/results_CAISE/a...
5,bpi_2012_CZ,ablate::time:timestamp,time:timestamp,/home/danbi/Projects/SANAGRAPH/results_CAISE/a...
6,bpi_2012_CZ,ablate::(case) AMOUNT_REQ,(case) AMOUNT_REQ,/home/danbi/Projects/SANAGRAPH/results_CAISE/a...
7,bpi_2012_CZ,Activity_zero,Activity,/home/danbi/Projects/SANAGRAPH/results_CAISE/a...
8,bpi_2013_CZ,full,,/home/danbi/Projects/SANAGRAPH/results_CAISE/b...
9,bpi_2013_CZ,ablate::Activity,Activity,/home/danbi/Projects/SANAGRAPH/results_CAISE/a...


The follwoing cell executes two checks:
1. The first one checks the correct number of runs.
2. The second one checks the metrics.
    1. `<target>_acc` for categorical attributes.
    2. `MAE_<target>` for numerical attributes.

The cell produces two outputs:
- `loaded_results`: CSV tables read.
- `log`: a row for file. With number of runs and targets.

In [10]:
loaded_results = []
inventory = []

for config in configurations.to_dict("records"):
    log = config["log"]
    excluded = config["excluded_feature"]

    info = dataset_features[log]

    categorical_targets = [
        target for target in info["categorical"]
        if target != excluded
    ]
    numerical_targets = [
        target for target in info["numerical"]
        if target != excluded
    ]

    expected_columns = {
        f"{target}_acc" for target in categorical_targets
    } | {
        f"MAE_{target}" for target in numerical_targets
    }

    for pattern in patterns:
        path = config["directory"] + f"{pattern}_V2_RESULTS_END.csv"
        table = pd.read_csv(path)

        assert len(table) == num_runs, (
            f"Unexpected number of runs: {path}"
        )

        actual_columns = {
            column for column in table.columns
            if column.endswith("_acc") or column.startswith("MAE_")
        }

        assert actual_columns == expected_columns, (
            f"Unexpected target metrics: {path}"
        )

        loaded_results.append({
            **config,
            "pattern": pattern,
            "path": path,
            "table": table,
        })

        inventory.append({
            "log": log,
            "model": config["model"],
            "pattern": pattern,
            "n_runs": len(table),
            "n_targets": len(expected_columns),
        })

inventory = pd.DataFrame(inventory)
inventory

,log,model,pattern,n_runs,n_targets
0,bpi_2012_CZ,full,odd,10,6
1,bpi_2012_CZ,full,even,10,6
2,bpi_2012_CZ,full,random,10,6
3,bpi_2012_CZ,full,window,10,6
4,bpi_2012_CZ,full,attr_level,10,6
...,...,...,...,...,...
200,BPI20_RequestForPayment_CZ,Activity_zero,odd,10,9
201,BPI20_RequestForPayment_CZ,Activity_zero,even,10,9
202,BPI20_RequestForPayment_CZ,Activity_zero,random,10,9
203,BPI20_RequestForPayment_CZ,Activity_zero,window,10,9


In [16]:
loaded_results

[{'log': 'bpi_2012_CZ',
  'model': 'full',
  'excluded_feature': '',
  'directory': '/home/danbi/Projects/SANAGRAPH/results_CAISE/bpi_2012_CZ/',
  'pattern': 'odd',
  'path': '/home/danbi/Projects/SANAGRAPH/results_CAISE/bpi_2012_CZ/odd_V2_RESULTS_END.csv',
  'table':    Activity_acc  org:resource_acc  concept:name_acc  lifecycle:transition_acc  \
  0      0.954275          0.778998          0.950486                  0.990183   
  1      0.950831          0.780332          0.954318                  0.988720   
  2      0.955825          0.777362          0.954620                  0.990011   
  3      0.948592          0.761646          0.953199                  0.989882   
  4      0.954275          0.778610          0.948506                  0.988074   
  5      0.948937          0.760355          0.948205                  0.987901   
  6      0.948721          0.770214          0.946956                  0.989882   
  7      0.950056          0.767330          0.955524                

Here transforms the tables in a single one. Every row contains only one measure, not like the original tables that have all the measurement in the same row.

In [12]:
records = []

for result in loaded_results:
    log = result["log"]
    excluded = result["excluded_feature"]
    table = result["table"]
    path = result["path"]

    info = dataset_features[log]

    targets = [
        (target, "categorical", "accuracy", f"{target}_acc")
        for target in info["categorical"]
        if target != excluded
    ]

    targets += [
        (target, "numerical", "mae", f"MAE_{target}")
        for target in info["numerical"]
        if target != excluded
    ]

    for target, kind, metric, column in targets:
        values = pd.to_numeric(table[column], errors="raise")

        assert np.isfinite(values).all(), (
            f"Non-finite values: {path}, {column}"
        )
        assert (values >= 0).all(), (
            f"Negative metric: {path}, {column}"
        )

        if metric == "accuracy":
            assert (values <= 1).all(), (
                f"Accuracy above 1: {path}, {column}"
            )

        for run, value in enumerate(values):
            records.append({
                "log": log,
                "model": result["model"],
                "excluded_feature": excluded,
                "pattern": result["pattern"],
                "run": run,
                "target": target,
                "kind": kind,
                "metric": metric,
                "value": value,
            })

runs = pd.DataFrame(records)
runs

,log,model,excluded_feature,pattern,run,target,kind,metric,value
0,bpi_2012_CZ,full,,odd,0,Activity,categorical,accuracy,0.954275
1,bpi_2012_CZ,full,,odd,1,Activity,categorical,accuracy,0.950831
2,bpi_2012_CZ,full,,odd,2,Activity,categorical,accuracy,0.955825
3,bpi_2012_CZ,full,,odd,3,Activity,categorical,accuracy,0.948592
4,bpi_2012_CZ,full,,odd,4,Activity,categorical,accuracy,0.954275
...,...,...,...,...,...,...,...,...,...
16695,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,5,case:RequestedAmount,numerical,mae,0.318445
16696,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,6,case:RequestedAmount,numerical,mae,0.319323
16697,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,7,case:RequestedAmount,numerical,mae,0.469204
16698,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,8,case:RequestedAmount,numerical,mae,0.383610


In [17]:
print(
    f"Checked {len(inventory)} result files; "
    f"{len(runs)} individual measurements."
)

inventory.groupby("log").agg(
    models=("model", "nunique"),
    files=("pattern", "size")
)

Checked 205 result files; 16700 individual measurements.


,models,files
log,,
BPI20_RequestForPayment_CZ,12,60
bpi_2012_CZ,8,40
bpi_2013_CZ,14,70
sp2020_CZ,7,35


### 3. Mean and standard deviation

Each row summarizes the 10 training for a target, model and pattern. It calculates the mean of the accuracy for categorical targets on the 10 runs, it does the same thing on numerical target but with the MAE and not the accuracy. All done mantaining fixed **dataset**, **model**, **pattern** and **target**. So, accuracy and MAE are not mixed.

In [18]:
summary = (runs.groupby(
    ["log", "model", "excluded_feature", "pattern", "target", "kind", "metric"],
    as_index=False, dropna=False,
)["value"].agg(mean="mean", std="std", n_runs="count"))
summary.head(20)

,log,model,excluded_feature,pattern,target,kind,metric,mean,std,n_runs
0,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:Activity,categorical,accuracy,0.977757,0.007080,10
1,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:OrganizationalEntity,categorical,accuracy,0.805749,0.027687,10
2,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:Project,categorical,accuracy,0.781423,0.041730,10
3,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:RequestedAmount,numerical,mae,0.335351,0.056371,10
4,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:RfpNumber,categorical,accuracy,0.037735,0.001241,10
5,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:Task,categorical,accuracy,0.760103,0.002143,10
6,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,org:resource,categorical,accuracy,0.966094,0.009688,10
7,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,org:role,categorical,accuracy,0.934279,0.008613,10
8,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,time:timestamp,numerical,mae,0.006941,0.000796,10
9,BPI20_RequestForPayment_CZ,Activity_zero,Activity,even,case:Activity,categorical,accuracy,1.000000,0.000000,10


### Differences compared to the full model

- Positive Delta = deterioration of performance.
- Negative Delta = increase of performace.



In [19]:
keys = ["log", "pattern", "target", "kind", "metric"]

- Accuracy: mean full minus mean variant. `delta_accuracy_pp` expresses the difference in percentile points.
- MAE: mean variant minus mean full.

In [ ]:
# take only the full model metrics
full = summary[summary.model == "full"][keys + ["mean", "std", "n_runs"]].rename(columns={"mean": "full_mean", "std": "full_std", "n_runs": "full_n_runs"})

In [21]:
full

,log,pattern,target,kind,metric,full_mean,full_std,full_n_runs
495,BPI20_RequestForPayment_CZ,attr_level,Activity,categorical,accuracy,0.919455,0.008418,10
496,BPI20_RequestForPayment_CZ,attr_level,case:Activity,categorical,accuracy,0.970215,0.010058,10
497,BPI20_RequestForPayment_CZ,attr_level,case:OrganizationalEntity,categorical,accuracy,0.832995,0.030449,10
498,BPI20_RequestForPayment_CZ,attr_level,case:Project,categorical,accuracy,0.811919,0.023466,10
499,BPI20_RequestForPayment_CZ,attr_level,case:RequestedAmount,numerical,mae,0.294827,0.043347,10
...,...,...,...,...,...,...,...,...
1665,sp2020_CZ,window,Activity,categorical,accuracy,0.738783,0.005366,10
1666,sp2020_CZ,window,DEVICETYPE,categorical,accuracy,0.997399,0.000491,10
1667,sp2020_CZ,window,REPAIR_IN_TIME_5D,categorical,accuracy,1.000000,0.000000,10
1668,sp2020_CZ,window,org:resource,categorical,accuracy,0.999688,0.000091,10


In [ ]:
# place alongside the variant metrics for comparison
comparison = summary[summary.model != "full"].rename(
    columns={"mean": "variant_mean", "std": "variant_std", "n_runs": "variant_n_runs"}
).merge(full, on=keys, how="left", validate="many_to_one")

In [24]:
comparison

,log,model,excluded_feature,pattern,target,kind,metric,variant_mean,variant_std,variant_n_runs,full_mean,full_std,full_n_runs
0,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:Activity,categorical,accuracy,0.977757,0.007080,10,0.970215,0.010058,10
1,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:OrganizationalEntity,categorical,accuracy,0.805749,0.027687,10,0.832995,0.030449,10
2,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:Project,categorical,accuracy,0.781423,0.041730,10,0.811919,0.023466,10
3,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:RequestedAmount,numerical,mae,0.335351,0.056371,10,0.294827,0.043347,10
4,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:RfpNumber,categorical,accuracy,0.037735,0.001241,10,0.037987,0.001310,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1500,sp2020_CZ,ablate::time:timestamp,time:timestamp,random,org:resource,categorical,accuracy,0.969122,0.001363,10,0.969065,0.002710,10
1501,sp2020_CZ,ablate::time:timestamp,time:timestamp,window,Activity,categorical,accuracy,0.742244,0.005483,10,0.738783,0.005366,10
1502,sp2020_CZ,ablate::time:timestamp,time:timestamp,window,DEVICETYPE,categorical,accuracy,0.998007,0.000320,10,0.997399,0.000491,10
1503,sp2020_CZ,ablate::time:timestamp,time:timestamp,window,REPAIR_IN_TIME_5D,categorical,accuracy,1.000000,0.000000,10,1.000000,0.000000,10


In [25]:
assert comparison.full_mean.notna().all()

In [26]:
comparison["delta"] = np.where(
    comparison.kind == "categorical", 
    comparison.full_mean - comparison.variant_mean,
    comparison.variant_mean - comparison.full_mean
)

In [27]:
comparison

,log,model,excluded_feature,pattern,target,kind,metric,variant_mean,variant_std,variant_n_runs,full_mean,full_std,full_n_runs,delta
0,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:Activity,categorical,accuracy,0.977757,0.007080,10,0.970215,0.010058,10,-0.007542
1,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:OrganizationalEntity,categorical,accuracy,0.805749,0.027687,10,0.832995,0.030449,10,0.027246
2,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:Project,categorical,accuracy,0.781423,0.041730,10,0.811919,0.023466,10,0.030497
3,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:RequestedAmount,numerical,mae,0.335351,0.056371,10,0.294827,0.043347,10,0.040524
4,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:RfpNumber,categorical,accuracy,0.037735,0.001241,10,0.037987,0.001310,10,0.000252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1500,sp2020_CZ,ablate::time:timestamp,time:timestamp,random,org:resource,categorical,accuracy,0.969122,0.001363,10,0.969065,0.002710,10,-0.000057
1501,sp2020_CZ,ablate::time:timestamp,time:timestamp,window,Activity,categorical,accuracy,0.742244,0.005483,10,0.738783,0.005366,10,-0.003461
1502,sp2020_CZ,ablate::time:timestamp,time:timestamp,window,DEVICETYPE,categorical,accuracy,0.998007,0.000320,10,0.997399,0.000491,10,-0.000608
1503,sp2020_CZ,ablate::time:timestamp,time:timestamp,window,REPAIR_IN_TIME_5D,categorical,accuracy,1.000000,0.000000,10,1.000000,0.000000,10,0.000000


In [28]:
comparison["delta_accuracy_pp"] = np.where(
    comparison.kind == "categorical", 100 * comparison.delta, np.nan
)

In [30]:
comparison.head(30)

,log,model,excluded_feature,pattern,target,kind,metric,variant_mean,variant_std,variant_n_runs,full_mean,full_std,full_n_runs,delta,delta_accuracy_pp
0,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:Activity,categorical,accuracy,0.977757,0.007080,10,0.970215,0.010058,10,-0.007542,-0.754151
1,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:OrganizationalEntity,categorical,accuracy,0.805749,0.027687,10,0.832995,0.030449,10,0.027246,2.724598
2,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:Project,categorical,accuracy,0.781423,0.041730,10,0.811919,0.023466,10,0.030497,3.049664
3,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:RequestedAmount,numerical,mae,0.335351,0.056371,10,0.294827,0.043347,10,0.040524,NaN
4,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:RfpNumber,categorical,accuracy,0.037735,0.001241,10,0.037987,0.001310,10,0.000252,0.025231
5,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,case:Task,categorical,accuracy,0.760103,0.002143,10,0.758507,0.001882,10,-0.001596,-0.159588
6,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,org:resource,categorical,accuracy,0.966094,0.009688,10,0.991329,0.003461,10,0.025235,2.523491
7,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,org:role,categorical,accuracy,0.934279,0.008613,10,0.967444,0.005368,10,0.033165,3.316542
8,BPI20_RequestForPayment_CZ,Activity_zero,Activity,attr_level,time:timestamp,numerical,mae,0.006941,0.000796,10,0.007622,0.001290,10,-0.000681,NaN
9,BPI20_RequestForPayment_CZ,Activity_zero,Activity,even,case:Activity,categorical,accuracy,1.000000,0.000000,10,1.000000,0.000000,10,0.000000,0.000000


In [32]:
activity_comparison = comparison[
    comparison.model.isin(["ablate::Activity", "Activity_zero"])
].copy()

In [36]:
inventory.to_csv(tables_dir + "inventory.csv", index=False)
runs.to_csv(tables_dir + "runs.csv", index=False)
summary.to_csv(tables_dir + "summary.csv", index=False)
comparison.to_csv(tables_dir + "comparison.csv", index=False)
activity_comparison.to_csv(tables_dir + "activity_comparison.csv", index=False)